In [1]:
import pandas as pd
import os
import numpy as np
import folium
import matplotlib.pyplot as plt
import textwrap

from matplotlib.backends.backend_pdf import PdfPages
from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime
from pyproj import Transformer

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

In [2]:
# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zu "Komplette_Datenbank" 
db_root = (
    base_dir.parent 
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken"
)

# Alle Unterordner lesen, Ordner mit "neustem" Datum setzen
timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

print(db_root)

C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken


In [3]:
# Ordner "Datenanalyse"
analysis_root = base_dir / "Rock-Type_Analysis"
analysis_root.mkdir(exist_ok=True)

# Zeitstempel erzeugen
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Neuen Ordner mit Zeitstempel erzeugen
output_pdf = analysis_root / timestamp
output_pdf.mkdir(parents=True, exist_ok=False)

print("Erzeugter Output-Ordner:")
print(output_pdf)

Erzeugter Output-Ordner:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Rock-Type_Analysis\Rock-Type_Analysis\2025-11-27_08-53-59


<div class="alert alert-info">
    <b>Erstellen einer .pdf zu allen gesammelten Datenbanken bzgl. ihres Rock Types</b><br>
    -> Wie viele Zeilen hat die .csv<br>
    -> In wie vielen Zeilen hat die Spalte "rock_type" Werte? (in %)<br>
    -> Welche Werte beinhaltet diese Spalte alles (Auflistung aller einzigartigen Attribute)
</div>

In [4]:
# --------- Alle .csv Dateien im angepassten Ordner sammeln ----------
csv_files = sorted(db_root.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"Keine .csv-Dateien in {db_root} gefunden.")

# --------- pdf Ziel definieren ---------
output_pdf = output_pdf / "Rocktype_Übersicht.pdf"

# --------- pdf mit einer Seite pro Datenbank ---------
with PdfPages(output_pdf) as pdf:
    for csv_path in csv_files:
        # -------- CSV einlesen -----------
        try:
            df = pd.read_csv(csv_path, low_memory=False)
        except Exception as e:
            # Falls eine Datei kaputt ist, Seite mit Fehlermeldung erzeugen
            fig = plt.figure(figsize=(8.27, 11.69))  # A4-Hochformat (ungefähr)
            fig.suptitle(f"Datei: {csv_path.name}", fontsize=14, fontweight="bold")
            fig.text(0.1, 0.8, f"Fehler beim Einlesen:\n{e}", fontsize=10)
            plt.axis("off")
            pdf.savefig(fig)
            plt.close(fig)
            continue

        total_rows = len(df)

        # -------- Prüfen, ob Spalte 'rock_type' existiert -----------
        if "rock_type" in df.columns:
            rock_col = df["rock_type"]
            non_null = rock_col.notna().sum()
            percent = (non_null / total_rows * 100) if total_rows > 0 else 0.0

            # ------------ Einzigartige Werte (ohne NaN) -----------
            unique_vals = sorted(rock_col.dropna().astype(str).unique())
        else:
            non_null = 0
            percent = 0.0
            unique_vals = []

        # ----------- Text für die Seite ------------
        header_lines = [
            f"Datei: {csv_path.name}",
            "",
            f"Anzahl Zeilen insgesamt: {total_rows}",
            f"Zeilen mit Wert in 'rock_type': {non_null} ({percent:0.2f} %)",
            "",
            "Einzigartige Werte in 'rock_type':"
        ]

        # --------- Einzigartige Werte als Liste ---------
        if unique_vals:
            # -------- Als kommagetrennte Zeichenkette ---------
            unique_str = ", ".join(unique_vals)
        else:
            unique_str = "(Keine Werte oder Spalte 'rock_type' nicht vorhanden.)"
 
        # ----------- Zeilen umbrechen, damit es auf die Seite passt -----------
        wrapped_unique_lines = textwrap.wrap(unique_str, width=100)

        # ------------ Neue Seite erstellen ------------
        fig = plt.figure(figsize=(8.27, 11.69))  
        fig.suptitle("Rock-Type-Analyse", fontsize=14, fontweight="bold", y=0.97)

        # ------------ Alle Zeilen gesammelt ausgeben -------------
        all_lines = header_lines + [""] + wrapped_unique_lines

        # --------- Text von oben nach unten ----------
        y_start = 0.9
        line_height = 0.025

        for i, line in enumerate(all_lines):
            y = y_start - i * line_height
            if y < 0.05:  # Abbruch, falls zu viele Zeilen
                fig.text(0.1, 0.05, "... (weitere Werte abgeschnitten)", fontsize=9)
                break
            fig.text(0.1, y, line, fontsize=10, va="top")

        plt.axis("off")
        pdf.savefig(fig)
        plt.close(fig)

print(f"PDF erstellt: {output_pdf}")

PDF erstellt: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Rock-Type_Analysis\Rock-Type_Analysis\2025-11-27_08-53-59\Rocktype_Übersicht.pdf
